In [17]:
import pandas as pd

In [18]:
# Cargar el dataset

file_path = '/content/drive/MyDrive/Colab Notebooks/SeminarioWeb/Reviews.csv'
df = pd.read_csv(file_path)
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [19]:
#Datos Nulos
df.isnull().sum()

,0
Id,0
ProductId,0
UserId,0
ProfileName,26
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,27
Text,0


In [20]:
# Eliminar duplicados
df = df.drop_duplicates()

In [21]:
# texto vacío en lugar de NaN
df['Text'] = df['Text'].fillna("")

# eliminar filas sin score (importante)
df = df.dropna(subset=['Score'])

In [22]:

#Limpiar texto
import re

def limpiar_texto(texto):
    texto = texto.lower()  # minúsculas
    texto = re.sub(r'<.*?>', '', texto)  # quitar HTML
    texto = re.sub(r'[^a-zA-Z\s]', '', texto)  # quitar símbolos
    texto = re.sub(r'\s+', ' ', texto).strip()  # espacios
    return texto

df['clean_text'] = df['Text'].apply(limpiar_texto)

Crear variables nuevas (feature engineering)
Longitud del texto

In [23]:
df['text_length'] = df['Text'].apply(len)

Ratio de utilidad

In [24]:
df['helpfulness_ratio'] = df['HelpfulnessNumerator'] / df['HelpfulnessDenominator']
df['helpfulness_ratio'] = df['helpfulness_ratio'].fillna(0)

Manejo de outliers

In [25]:
df['text_length'].describe()

,text_length
count,568454.000000
mean,436.222083
std,445.339741
min,12.000000
25%,179.000000
50%,302.000000
75%,527.000000
max,21409.000000


Puedes filtrar textos demasiado largos:

In [26]:
df = df[df['text_length'] < 5000]

🔹 8. Convertir fechas

In [27]:
df['Time'] = pd.to_datetime(df['Time'], unit='s')

Crear sentimiento

In [28]:
!pip install vaderSentiment

In [15]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df['sentiment'] = df['clean_text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])

Validación final

In [16]:
df.isnull().sum()
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,clean_text,text_length,helpfulness_ratio,sentiment
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,2011-04-27,Good Quality Dog Food,I have bought several of the Vitality canned d...,i have bought several of the vitality canned d...,263,1.0,0.9441
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,2012-09-07,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,product arrived labeled as jumbo salted peanut...,190,0.0,-0.5664
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,2008-08-18,"""Delight"" says it all",This is a confection that has been around a fe...,this is a confection that has been around a fe...,509,1.0,0.8138
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,2011-06-13,Cough Medicine,If you are looking for the secret ingredient i...,if you are looking for the secret ingredient i...,219,1.0,0.4404
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,2012-10-21,Great taffy,Great taffy at a great price. There was a wid...,great taffy at a great price there was a wide ...,140,0.0,0.9468


In [29]:
df.to_csv("reviews_limpio.csv", index=False)